In [30]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import os
from pathlib import Path
import sqlite3

In [31]:
#Conectar con la base de datos y cargar las 3 tablas en dataframes
sqlite_path = Path("../datos/intermedios/mercado_inmobiliario_bogota.sqlite")
conn = sqlite3.connect(sqlite_path)
m2_bogota = pd.read_sql_query("SELECT * FROM m2_bogota", conn)
listings = pd.read_sql_query("SELECT * FROM listings", conn)
listings_det = pd.read_sql_query("SELECT * FROM listings_det", conn)

In [32]:
# merje inner join entre listings y listings_det
df_listings = pd.merge(listings, listings_det, on='id', how='inner')


In [33]:
#merge entre df_listings y m2_bogota left join para que no se pierdan registros de df_listings
df = pd.merge(df_listings, m2_bogota, on='neighborhood', how='left')

In [34]:
#renombrar price a precio_noche y precio a precio_m2
df.rename(columns={'price': 'precio_noche', 'precio': 'precio_m2'}, inplace=True)

#Eliminar columna id
df.drop(columns=['id'], inplace=True)

In [35]:
tasa_uso = 0.5

condiciones = [
    df["room_type"].eq("Entire home/apt"),
    df["room_type"].eq("Private room"),
    df["room_type"].eq("Shared room")
]

valores = [
    df["precio_noche"],
    df["precio_noche"] * df["bedrooms"] * tasa_uso,
    df["precio_noche"] * df["accommodates"] * tasa_uso
]

df["precio_noche_Total"] = np.select(
    condiciones,
    valores,
    default=np.nan
)

In [36]:
#Crar una variable ingreso_anual como multiplicación de precio_noche_Total por estimated_occupancy_1365d 
df["ingreso_anual"] = df["precio_noche_Total"] * df["estimated_occupancy_l365d"]

In [ ]:
condiciones_m2 = [

    df["bedrooms"].ge(4),
    df["bedrooms"].eq(3) & df["bathrooms"].gt(2),
    df["bedrooms"].eq(3),
    df["bedrooms"].eq(2),
    df["bedrooms"].eq(1)

]

valores_m2 = [140, 110, 90, 65, 50]

df["m2_estimados"] = np.select(
    condiciones_m2,
    valores_m2,
    default=np.nan

)

In [43]:
#crear variable costo_adquisicion como multiplicación de m2_estimados por precio_m2
df["costo_adquisicion"] = df["m2_estimados"] * df["precio_m2"]

In [64]:
#discretizar la variable bedrooms en 4 categorías: 1, 2, 3, 4+

df["bedrooms_disc"] = pd.cut(

    df["bedrooms"],

    bins=[1, 2, 3, 4, np.inf],

    labels=["1_hab", "2_hab", "3_hab", "4+_hab"],

    right=False

)


In [63]:
#discretizar la variable bathrooms para que sea menor o igual a 1, entre 1 y 2 y mas de 2
df["bathrooms_disc"] = pd.cut(
    df["bathrooms"],
    bins=[0, 1, 2, float("inf")],
    labels=["01_Uno", "03_Dos", "03_Mas_de_dos"],
    right=True,
    include_lowest=True
)

In [69]:
#discretizar la variable accommodates en 4 categorías: 1, 2, 3, 4, 4+
df["accommodates_disc"] = pd.cut(
    df["accommodates"],
    bins=[0, 1, 2, 3, 4, float("inf")],
    labels=["01_Uno", "02_Dos", "03_Tres", "04_Cuatro", "05_mas_cuatro"],
    right=True,
    include_lowest=True
)

In [74]:
#discretizar variable beds en 4 categorías: 1, 2, 3, 4, 4+
df["beds_disc"] = pd.cut(
    df["beds"],
    bins=[0, 1, 2, 3, 4, float("inf")],
    labels=["01_cama", "02_camas", "03_camas", "04_camas", "05_mas_camas"],
    right=True,
    include_lowest=True
)

In [77]:
#crear nueva tabla en base de datos que se llame tabla_analitica con el dataframe df
df.to_sql("tabla_analitica", conn, if_exists="replace", index=False)

9687